# DDRI 대표 15개 상위 5개 오류 스테이션 피처 연결 및 subset 우선순위 정리

- 목적: 대표 15개 상위 오류 5개 스테이션을 군집별 권장 보강 피처와 직접 연결하고, 다음 subset 실험 우선순위를 정리한다.
- 입력:
  - `rep15_error_analysis/output/data/ddri_rep15_top5_station_error_summary.csv`
  - `rep15_error_analysis/output/data/ddri_rep15_top5_station_peak_error_hours.csv`
- 출력:
  - `rep15_error_analysis/output/data/ddri_rep15_top5_feature_linkage_table.csv`
  - `rep15_error_analysis/output/data/ddri_cluster_subset_priority_table.csv`


In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
ANALYSIS_DIR = ROOT / 'works/05_prediction_long/cheng80/rep15_error_analysis'
OUTPUT_DATA_DIR = ANALYSIS_DIR / 'output/data'
TOP5_SUMMARY_PATH = OUTPUT_DATA_DIR / 'ddri_rep15_top5_station_error_summary.csv'
TOP5_PEAK_PATH = OUTPUT_DATA_DIR / 'ddri_rep15_top5_station_peak_error_hours.csv'

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
top5_summary = pd.read_csv(TOP5_SUMMARY_PATH, encoding='utf-8-sig')
peak_hours = pd.read_csv(TOP5_PEAK_PATH, encoding='utf-8-sig')

top5_summary['station_id'] = top5_summary['station_id'].astype(str)
peak_hours['station_id'] = peak_hours['station_id'].astype(str)

peak_hour_map = peak_hours.groupby('station_id')['hour'].apply(lambda s: ', '.join(str(int(v)) for v in s.tolist())).to_dict()

cluster_feature_map = {
    '아침 도착 업무 집중형': {
        'cluster_code': 'cluster01',
        'feature_bundle': 'is_commute_hour, is_after_holiday, subway_distance_m, bus_stop_count_300m, rolling_mean_6h, rolling_mean_12h, rolling_std_6h, rolling_std_12h, lag_48h, commute_morning_flag, commute_evening_flag',
        'cluster_priority_reason': '2차와 3차에서 가장 분명한 개선이 확인됐고, 대표 상위 오류 1·2위가 모두 이 군집이다',
        'subset_priority': 1,
        'subset_priority_label': '즉시 진행',
    },
    '주거 도착형': {
        'cluster_code': 'cluster02',
        'feature_bundle': 'is_night_hour, is_weekend, is_holiday_eve, heavy_rain_flag, station_elevation_m, bus_stop_count_300m',
        'cluster_priority_reason': '대표 상위 오류 3위가 포함되고 아침 시간대 과소예측 패턴이 뚜렷해 경량 subset 검증 가치가 있다',
        'subset_priority': 2,
        'subset_priority_label': '우선 검토',
    },
    '업무/상업 혼합형': {
        'cluster_code': 'cluster00',
        'feature_bundle': 'is_commute_hour, is_after_holiday, subway_distance_m, bus_stop_count_300m, restaurant_count_300m, cafe_count_300m, rolling_mean_6h',
        'cluster_priority_reason': '대표 상위 오류 4위가 포함되지만 2차 개선폭이 작아 cluster02 다음 순서가 적절하다',
        'subset_priority': 3,
        'subset_priority_label': '후속 검토',
    },
    '외곽 주거형': {
        'cluster_code': 'cluster04',
        'feature_bundle': 'station_elevation_m, elevation_diff_nearest_subway_m, distance_naturepark_m, distance_river_boundary_m, bus_stop_count_300m',
        'cluster_priority_reason': '대표 상위 오류 5위가 있으나 전체 개선폭이 작아 우선순위는 낮다',
        'subset_priority': 4,
        'subset_priority_label': '보류 검토',
    },
}

station_pattern_map = {
    '2377': {
        'pattern_type': '저녁 피크 변동형',
        'feature_focus': 'is_commute_hour, commute_evening_flag, rolling_mean_6h, rolling_mean_12h, bus_stop_count_300m',
        'interpretation_note': '16~20시 저녁 피크 시간대 변동이 커서 퇴근 시간대 직접 표시와 단기 추세 보강이 필요하다',
    },
    '2348': {
        'pattern_type': '저녁 피크 과대예측형',
        'feature_focus': 'is_commute_hour, commute_evening_flag, is_after_holiday, rolling_std_6h, subway_distance_m',
        'interpretation_note': '16~19시 과대예측과 11시 오차가 함께 보여 업무권 피크와 낮 시간대 구조를 동시에 봐야 한다',
    },
    '4917': {
        'pattern_type': '아침 과소예측형',
        'feature_focus': 'is_night_hour, is_weekend, is_holiday_eve, bus_stop_count_300m',
        'interpretation_note': '05~08시와 17시에 실제값이 더 높아 주거형 아침 이동과 생활 리듬을 직접 반영할 필요가 있다',
    },
    '2328': {
        'pattern_type': '업무권 혼합 시간대형',
        'feature_focus': 'is_commute_hour, restaurant_count_300m, cafe_count_300m, rolling_mean_6h',
        'interpretation_note': '08시, 11시, 16~18시가 섞여 있어 출퇴근과 점심 상권 흐름이 같이 반영되는 구조로 보인다',
    },
    '2359': {
        'pattern_type': '외곽 생활 이동형',
        'feature_focus': 'station_elevation_m, distance_naturepark_m, distance_river_boundary_m, bus_stop_count_300m',
        'interpretation_note': '08시와 16~18시가 동시에 높아 외곽 주거형의 생활권 이동 시간대를 함께 봐야 한다',
    },
}

rows = []
for _, row in top5_summary.iterrows():
    sid = row['station_id']
    cluster_name = row['station_group']
    cinfo = cluster_feature_map[cluster_name]
    sinfo = station_pattern_map[sid]
    rows.append({
        'analysis_rank': int(row['analysis_rank']),
        'station_id': sid,
        'station_name': row['station_name'],
        'station_group': cluster_name,
        'cluster_code': cinfo['cluster_code'],
        'rmse': row['rmse'],
        'mae': row['mae'],
        'bias_mean': row['bias_mean'],
        'peak_error_hours': peak_hour_map.get(sid, ''),
        'pattern_type': sinfo['pattern_type'],
        'feature_focus': sinfo['feature_focus'],
        'cluster_feature_bundle': cinfo['feature_bundle'],
        'interpretation_note': sinfo['interpretation_note'],
        'subset_priority': cinfo['subset_priority'],
        'subset_priority_label': cinfo['subset_priority_label'],
    })

feature_linkage_df = pd.DataFrame(rows).sort_values(['subset_priority', 'analysis_rank']).reset_index(drop=True)

cluster_priority_df = feature_linkage_df.groupby(['subset_priority', 'subset_priority_label', 'cluster_code', 'station_group'], as_index=False).agg(
    representative_station_count=('station_id', 'count'),
    representative_station_ids=('station_id', lambda s: ', '.join(s.tolist())),
    avg_rmse=('rmse', 'mean'),
    avg_mae=('mae', 'mean'),
    peak_time_signature=('peak_error_hours', 'first'),
    feature_bundle=('cluster_feature_bundle', 'first'),
)
cluster_priority_df['priority_reason'] = cluster_priority_df['station_group'].map({
    k: v['cluster_priority_reason'] for k, v in cluster_feature_map.items()
})
cluster_priority_df = cluster_priority_df.sort_values(['subset_priority', 'avg_rmse'], ascending=[True, False]).reset_index(drop=True)

feature_linkage_path = OUTPUT_DATA_DIR / 'ddri_rep15_top5_feature_linkage_table.csv'
cluster_priority_path = OUTPUT_DATA_DIR / 'ddri_cluster_subset_priority_table.csv'

feature_linkage_df.to_csv(feature_linkage_path, index=False, encoding='utf-8-sig')
cluster_priority_df.to_csv(cluster_priority_path, index=False, encoding='utf-8-sig')

print(f'saved: {feature_linkage_path}')
print(f'saved: {cluster_priority_path}')


saved: /Users/cheng80/Desktop/ddri_work/works/05_prediction_long/cheng80/rep15_error_analysis/output/data/ddri_rep15_top5_feature_linkage_table.csv
saved: /Users/cheng80/Desktop/ddri_work/works/05_prediction_long/cheng80/rep15_error_analysis/output/data/ddri_cluster_subset_priority_table.csv


In [3]:
feature_linkage_df


,analysis_rank,station_id,station_name,station_group,cluster_code,rmse,mae,bias_mean,peak_error_hours,pattern_type,feature_focus,cluster_feature_bundle,interpretation_note,subset_priority,subset_priority_label
0,1,2377,수서역 5번출구,아침 도착 업무 집중형,cluster01,1.716840,1.184462,-0.107803,"18, 19, 17, 20, 16",저녁 피크 변동형,"is_commute_hour, commute_evening_flag, rolling...","is_commute_hour, is_after_holiday, subway_dist...",16~20시 저녁 피크 시간대 변동이 커서 퇴근 시간대 직접 표시와 단기 추세 보강...,1,즉시 진행
1,2,2348,포스코사거리(기업은행),아침 도착 업무 집중형,cluster01,1.690407,1.053046,-0.385833,"18, 17, 16, 19, 11",저녁 피크 과대예측형,"is_commute_hour, commute_evening_flag, is_afte...","is_commute_hour, is_after_holiday, subway_dist...",16~19시 과대예측과 11시 오차가 함께 보여 업무권 피크와 낮 시간대 구조를 동...,1,즉시 진행
2,3,4917,일원에코파크 주차장,주거 도착형,cluster02,1.275094,0.834525,0.300213,"8, 7, 17, 5, 6",아침 과소예측형,"is_night_hour, is_weekend, is_holiday_eve, bus...","is_night_hour, is_weekend, is_holiday_eve, hea...",05~08시와 17시에 실제값이 더 높아 주거형 아침 이동과 생활 리듬을 직접 반영...,2,우선 검토
3,4,2328,르네상스 호텔 사거리 역삼지하보도 7번출구 앞,업무/상업 혼합형,cluster00,0.946361,0.671325,-0.112874,"17, 18, 11, 16, 8",업무권 혼합 시간대형,"is_commute_hour, restaurant_count_300m, cafe_c...","is_commute_hour, is_after_holiday, subway_dist...","08시, 11시, 16~18시가 섞여 있어 출퇴근과 점심 상권 흐름이 같이 반영되는...",3,후속 검토
4,5,2359,"국립국악중,고교 정문 맞은편",외곽 주거형,cluster04,0.939339,0.668261,-0.027688,"8, 16, 18, 17, 20",외곽 생활 이동형,"station_elevation_m, distance_naturepark_m, di...","station_elevation_m, elevation_diff_nearest_su...",08시와 16~18시가 동시에 높아 외곽 주거형의 생활권 이동 시간대를 함께 봐야 한다,4,보류 검토


In [4]:
cluster_priority_df


,subset_priority,subset_priority_label,cluster_code,station_group,representative_station_count,representative_station_ids,avg_rmse,avg_mae,peak_time_signature,feature_bundle,priority_reason
0,1,즉시 진행,cluster01,아침 도착 업무 집중형,2,"2377, 2348",1.703623,1.118754,"18, 19, 17, 20, 16","is_commute_hour, is_after_holiday, subway_dist...","2차와 3차에서 가장 분명한 개선이 확인됐고, 대표 상위 오류 1·2위가 모두 이 ..."
1,2,우선 검토,cluster02,주거 도착형,1,4917,1.275094,0.834525,"8, 7, 17, 5, 6","is_night_hour, is_weekend, is_holiday_eve, hea...",대표 상위 오류 3위가 포함되고 아침 시간대 과소예측 패턴이 뚜렷해 경량 subse...
2,3,후속 검토,cluster00,업무/상업 혼합형,1,2328,0.946361,0.671325,"17, 18, 11, 16, 8","is_commute_hour, is_after_holiday, subway_dist...",대표 상위 오류 4위가 포함되지만 2차 개선폭이 작아 cluster02 다음 순서가...
3,4,보류 검토,cluster04,외곽 주거형,1,2359,0.939339,0.668261,"8, 16, 18, 17, 20","station_elevation_m, elevation_diff_nearest_su...",대표 상위 오류 5위가 있으나 전체 개선폭이 작아 우선순위는 낮다
